# Tutorial 19: Object Stores Beyond AWS

LAILA's S3 adapter is the most commonly seen one, but the same `memorize` / `remember` / `forget` API targets Google Cloud Storage, Azure Blob Storage, Cloudflare R2, and Backblaze B2 with one-line constructor changes. This tutorial walks through each.

You will:

- Read provider credentials from a `secrets.toml`
- Instantiate one pool per provider you have access to
- Round-trip a small payload through each
- Compare the constructors side-by-side

**Prerequisites:** `pip install "laila-core[gcs,azure,cloudflare,backblaze]"` plus credentials for whichever provider you want to try. The notebook wraps each provider in try/except so missing credentials skip cleanly.

In [ ]:
import numpy as np

import laila
from laila.pool import GCSPool, AzurePool, CloudflarePool, BackblazePool

## Credentials

Drop a `secrets.toml` next to the notebook with whichever providers you want to test. Every provider's keys are optional — the cells below detect missing values and skip.

```toml
# secrets.toml
GCS_PROJECT_ID = "my-gcp-project"
GCS_BUCKET = "my-gcs-bucket"
# GCS_SERVICE_ACCOUNT_JSON points at a service account JSON file path

AZURE_CONNECTION_STRING = "DefaultEndpointsProtocol=..."
AZURE_CONTAINER = "my-container"

R2_ACCOUNT_ID = "..."
R2_ACCESS_KEY_ID = "..."
R2_SECRET_ACCESS_KEY = "..."
R2_BUCKET = "my-r2-bucket"

B2_APP_KEY_ID = "..."
B2_APP_KEY = "..."
B2_BUCKET = "my-b2-bucket"
```

In [ ]:
import os
if os.path.exists("./secrets.toml"):
    laila.read_args("./secrets.toml")
    print("loaded ./secrets.toml")
else:
    print("no secrets.toml found - cells below will skip if creds are missing")

## A common workload

We round-trip the same numpy array through each provider so the only difference between cells is the constructor.

In [ ]:
def round_trip(pool, label):
    laila.memory.extend(pool, pool_nickname=label)
    e = laila.constant(data=np.arange(16).reshape(4, 4), nickname=f"{label}_demo")
    laila.memorize(e, pool_nickname=label).wait()
    recovered = laila.remember(nickname=f"{label}_demo", pool_nickname=label, persist=False).wait()
    if isinstance(recovered, list):
        recovered = recovered[0]
    laila.forget(nickname=f"{label}_demo", pool_nickname=label).wait()
    print(f"  {label}: shape={recovered.data.shape}")

## Google Cloud Storage

In [ ]:
try:
    import json
    sa_path = laila.args.get("GCS_SERVICE_ACCOUNT_JSON") if hasattr(laila.args, "get") else None
    sa_info = json.loads(open(sa_path).read()) if sa_path else None
    gcs = GCSPool(
        service_account_info=sa_info,
        project_id=laila.args.GCS_PROJECT_ID,
        bucket_name=laila.args.GCS_BUCKET,
        nickname="gcs",
    )
    round_trip(gcs, "gcs")
except Exception as e:
    print("  gcs skipped:", type(e).__name__, "-", e)

## Azure Blob Storage

In [ ]:
try:
    azure = AzurePool(
        connection_string=laila.args.AZURE_CONNECTION_STRING,
        container_name=laila.args.AZURE_CONTAINER,
        nickname="azure",
    )
    round_trip(azure, "azure")
except Exception as e:
    print("  azure skipped:", type(e).__name__, "-", e)

## Cloudflare R2

R2 speaks the S3 protocol — the `CloudflarePool` is a thin wrapper that fills in the endpoint URL from your account id.

In [ ]:
try:
    r2 = CloudflarePool(
        account_id=laila.args.R2_ACCOUNT_ID,
        access_key_id=laila.args.R2_ACCESS_KEY_ID,
        secret_access_key=laila.args.R2_SECRET_ACCESS_KEY,
        bucket_name=laila.args.R2_BUCKET,
        nickname="r2",
    )
    round_trip(r2, "r2")
except Exception as e:
    print("  r2 skipped:", type(e).__name__, "-", e)

## Backblaze B2

B2 also speaks S3-compatible; the constructor takes application keys instead of access keys.

In [ ]:
try:
    b2 = BackblazePool(
        application_key_id=laila.args.B2_APP_KEY_ID,
        application_key=laila.args.B2_APP_KEY,
        bucket_name=laila.args.B2_BUCKET,
        nickname="b2",
    )
    round_trip(b2, "b2")
except Exception as e:
    print("  b2 skipped:", type(e).__name__, "-", e)

## Picking a provider

| Provider | Strengths | Notes |
|---|---|---|
| AWS S3 | Mature ecosystem, broadest region coverage | The default in Tutorials 3+ |
| GCS | Native integration with GCP services, strong throughput | Service-account JSON is the standard credential |
| Azure Blob | Tight Azure / Entra integration | Connection strings collapse host + key into one value |
| Cloudflare R2 | Zero egress fees within Cloudflare's network | S3-compatible; works with most existing tooling |
| Backblaze B2 | Cheapest storage tier among major providers | S3-compatible; pair with B2 native API for lifecycle rules |

## Summary

- The body of every `memorize` / `remember` call is identical across providers — only the constructor changes.
- Credentials flow through `laila.args` via `read_args`, so a `secrets.toml` works for all providers at once.
- Wrap each provider cell in try/except so missing credentials skip gracefully.